In [1]:
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

con.sql("""
    CREATE OR REPLACE SECRET minio (
        TYPE s3,
        KEY_ID 'minioadmin',
        SECRET 'minioadmin123',
        ENDPOINT 'minio:9000',
        URL_STYLE 'path',
        USE_SSL false
    );
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [4]:
con.sql("""
    SELECT order_status, count(*) AS n
    FROM read_csv('s3://raw-data/orders.csv')
    GROUP BY order_status
    ORDER BY n DESC;
""").show();

┌─────────────────┬───────┐
│  order_status   │   n   │
│     varchar     │ int64 │
├─────────────────┼───────┤
│ COMPLETE        │ 22899 │
│ PENDING_PAYMENT │ 15030 │
│ PROCESSING      │  8275 │
│ PENDING         │  7610 │
│ CLOSED          │  7556 │
│ ON_HOLD         │  3798 │
│ SUSPECTED_FRAUD │  1558 │
│ CANCELED        │  1428 │
│ PAYMENT_REVIEW  │   729 │
└─────────────────┴───────┘



In [5]:
con.sql("""
COPY (
    SELECT * FROM read_csv('s3://raw-data/orders.csv')
)
TO 's3://processed/retail_db/orders'
(FORMAT parquet, PARTITION_BY (order_status), OVERWRITE_OR_IGNORE);
""");

In [7]:
con.sql("""
DESCRIBE SELECT *
FROM read_parquet('s3://processed/retail_db/orders/order_status=COMPLETE/*.parquet',
                  hive_partitioning = false);
""").show()

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ order_date        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_customer_id │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [3]:
import duckdb

con = duckdb.connect()
con.sql("INSTALL mysql; LOAD mysql;")

con.sql("""
ATTACH 'host=mysql-retail user=root password=rootpass database=retail_db'
    AS retail (TYPE mysql);
""");

con.sql("""
    CREATE OR REPLACE SECRET minio (
        TYPE s3,
        KEY_ID 'minioadmin',
        SECRET 'minioadmin123',
        ENDPOINT 'minio:9000',
        URL_STYLE 'path',
        USE_SSL false
    );
""")

con.sql("""
    COPY (SELECT * FROM retail.customers)
    TO 's3://processed/retail_db/customers'
    (FORMAT parquet, PARTITION_BY (customer_state), OVERWRITE_OR_IGNORE);
""");